[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/building-rag-pipelines/blob/main/notebooks/02_loading.ipynb)

# Building RAG Pipelines
## Notebook 02: Loading — Data Ingestion
**Duration:** 15 min &nbsp;|&nbsp; **Mode:** Conceptual + Demonstration

> We build RAG as a **modular pipeline**. Every stage is taught **WHY → WHAT → HOW**,
> and we keep asking the session's guiding question: *"What happens if this step is
> poorly designed?"* Frameworks (LangChain) appear only as a **parallel mapping** —
> they abstract the mechanics but **do not eliminate the design decisions**.

![pipeline](https://dummyimage.com/1000x70/1f2937/ffffff&text=Loading+%E2%86%92+Chunking+%E2%86%92+Retrieval+%E2%86%92+Augmentation+%E2%86%92+Generation+%E2%86%92+Evaluation)

In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first.
# ============================================================
# It (1) installs dependencies, (2) makes the `rag_pipeline` package importable,
# and (3) locates the sample corpus. Everything below runs even with NO API key,
# because the package falls back to a deterministic offline "mock" provider.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

# >>> INSTRUCTOR: set this to your repo URL so Colab can fetch the package. <<<
REPO_URL = "https://github.com/baluragala/building-rag-pipelines.git"

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# Core deps. `openai`+`tiktoken` enable the real stack; the rest power loaders/retrieval.
_pip("numpy", "openai", "tiktoken", "rank-bm25", "beautifulsoup4", "pypdf",
     "langchain-community", "langchain-text-splitters", "langchain-openai", "faiss-cpu")

# Make `rag_pipeline` importable.
try:
    import rag_pipeline  # already on the path (local run, or repo already cloned)
except ModuleNotFoundError:
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
        if os.path.isdir("building-rag-pipelines"):
            sys.path.insert(0, "building-rag-pipelines")
        else:
            print("Clone failed. Upload the `rag_pipeline/` folder and `data/` via the "
                  "Colab file browser (left panel), then re-run this cell.")
    else:
        sys.path.insert(0, os.path.abspath(".."))  # notebooks/ -> repo root
    import rag_pipeline

def data_path(*parts):
    for base in ("data", "../data", "building-rag-pipelines/data"):
        p = os.path.join(base, *parts)
        if os.path.exists(p):
            return p
    return os.path.join("data", *parts)

print("rag_pipeline", rag_pipeline.__version__, "ready.  Colab:", IN_COLAB)

In [ ]:
# ============================================================
# CHOOSE YOUR PROVIDERS  (OpenAI is the default)
# ============================================================
# Default stack = OpenAI: gpt-4o-mini (LLM) + text-embedding-3-small (embeddings).
# In Colab the key is read automatically from the Colab SECRETS manager:
#   left sidebar -> key icon -> add a secret named OPENAI_API_KEY
#   -> toggle "Notebook access" ON  -> re-run this cell.
# If no key is found anywhere, we fall back to the offline MOCK so the notebook
# still runs end-to-end.
import os

def _load_openai_key():
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass  # not in Colab, secret missing, or access not granted
    return False

if _load_openai_key():
    os.environ.setdefault("RAG_LLM_PROVIDER", "openai")
    os.environ.setdefault("RAG_EMBED_PROVIDER", "openai")
    print("OpenAI key found -> using the OpenAI stack.")
else:
    os.environ["RAG_LLM_PROVIDER"] = "mock"
    os.environ["RAG_EMBED_PROVIDER"] = "mock"
    print("No OPENAI_API_KEY found -> using the offline MOCK providers.\n"
          "In Colab: add a Secret named OPENAI_API_KEY (key icon, left sidebar),\n"
          "enable Notebook access, and re-run this cell to switch to OpenAI.")

from rag_pipeline import config
print(config.current_config())

## WHY — retrieval can only surface what ingestion let in

Loading is the quiet stage everyone skips, and it is where a shocking share of
RAG failures are *born*. A PDF loaded as one giant blob with page numbers and
headers glued into the sentences will chunk badly, embed noisily, and retrieve
the wrong spans. The model never had a chance.

> **What happens if this step is poorly designed?** Garbage in → garbage
> retrieved → confident garbage out. And because the damage is upstream, you'll
> waste days tuning retrieval and prompts that were never the problem.

## WHAT — the do's & don'ts of ingestion

**DO**
- **Clean** text (normalise whitespace/encoding, strip page furniture).
- **Preserve structure** (headings, paragraphs) — it's semantic signal chunking will use.
- **Attach metadata** (source, title, page) — required for citations & metadata filtering later.

**DON'T**
- Dump raw bytes or ignore encoding/noise.
- Throw away *where a span came from* — you can never add provenance back later.

Formats you'll meet: **PDF, HTML, Markdown, APIs**. Each needs a loader that knows
its quirks (PDFs have page furniture; HTML has nav/script noise; APIs return JSON).

## HOW (from scratch) — every loader attaches metadata

Our `Document` is just `page_content` + `metadata` (the same shape LangChain uses).
That single discipline — *always carry metadata* — is the whole point of this stage.

In [ ]:
from rag_pipeline.loaders import (
    load_markdown, load_html, load_directory, clean_text, Document
)

# Markdown loader captures the H1 as the title and keeps heading structure.
md_doc = load_markdown(data_path("corpus", "acme_pricing.md"))[0]
print("METADATA:", md_doc.metadata)
print("\nFIRST 200 CHARS:\n", md_doc.page_content[:200])

> ### ✋ Predict before you run
> The FAQ file `acme_faq.html` contains a <nav> bar ('Home | Docs | Pricing | Contact') and a <footer>. After we load it with the HTML loader, will that navigation text appear in `page_content`? Should it?
>
> *Write your guess down before executing the next cell. The gap between your
> prediction and the result is where the learning happens.*

In [ ]:
# HTML loader drops <script>/<style>/<nav>/<footer> boilerplate — DON'T embed navigation noise.
html_doc = load_html(data_path("corpus", "acme_faq.html"))[0]
print("TITLE:", html_doc.metadata["title"])
print("\nCLEANED TEXT (note: no 'Home | Docs | Pricing' nav, no footer):\n")
print(html_doc.page_content[:400])

### The cleaning step, made visible

`clean_text` normalises line endings, collapses excess whitespace, and strips
common PDF artefacts (form-feeds, isolated page numbers) **while preserving
paragraph breaks** (those breaks are boundaries the recursive chunker needs).

In [ ]:
dirty = "Acme  Cloud\r\n\n\n\n  Pricing   Guide  \x0c   12   \nStarter costs $29."
print("RAW  :", repr(dirty))
print("CLEAN:", repr(clean_text(dirty)))

### PDFs — the format most likely to sabotage retrieval

PDFs are where loading most often goes wrong. They carry **page furniture** —
running headers/footers, "CONFIDENTIAL" stamps, isolated page numbers — glued
into the text, and they have real **page boundaries**.

`load_pdf(..., per_page=True)` returns **one `Document` per page** with a `page`
number in metadata, so you keep page-level provenance for citations. `clean_text`
then strips the isolated page numbers (but deliberately leaves the header text —
cleaning is a *design choice* you tune per source, not a magic button).

In [ ]:
# Load a 2-page sample PDF. One Document per page, with a `page` number in metadata.
from rag_pipeline.loaders import load_pdf

pdf_docs = load_pdf(data_path("samples", "acme_onepager.pdf"))   # per_page=True by default
print(f"{len(pdf_docs)} page-documents loaded from the PDF.\n")
for d in pdf_docs:
    print(f"page {d.metadata['page']}  ({len(d.page_content)} chars)  "
          f"source={d.metadata['source'].split('/')[-1]}")
    print("   ", repr(d.page_content[:90].strip()), "\n")

Notice the `page` number rode along in metadata — that's what lets a later answer
cite *"Acme Overview, p.2"*. **What happens if you'd loaded the whole PDF as one
blob instead?** You'd lose page provenance, and the header/footer noise on every
page would repeat through your chunks, polluting the embeddings.

In [ ]:
# Load the whole corpus at once — load_directory dispatches by file extension.
docs = load_directory(data_path("corpus"))
print(f"Loaded {len(docs)} documents:")
for d in docs:
    print(f"  {d.metadata['title']:<40} loader={d.metadata['loader']:<15} {len(d.page_content)} chars")

## HOW (parallel mapping) — LangChain DocumentLoaders

The framework wraps the *same* steps: read → produce `Document`s with
`page_content` + `metadata`. **Flexibility vs abstraction:** hand-rolled loaders
give you total control over cleaning and metadata; DocumentLoaders save
boilerplate but **you still decide** what to clean and what metadata to keep —
the framework does not make that design decision for you.

In [ ]:
# Parallel mapping (requires langchain-community; safe to skip if not installed).
try:
    from rag_pipeline.loaders import load_directory_langchain
    lc_docs = load_directory_langchain(data_path("corpus"))
    print(f"LangChain returned {len(lc_docs)} docs; same (page_content, metadata) shape.")
    print("Example metadata:", lc_docs[0].metadata)
except Exception as e:
    print("LangChain not installed — concept still holds:", e)

## Recap
- Loading decides the ceiling on everything downstream.
- **DO**: clean, preserve structure, attach metadata. **DON'T**: raw dumps, drop provenance.
- Framework or not, *you* own the cleaning & metadata design.

**Next → Notebook 03 (Chunking):** how we slice these documents into retrievable
units — and how the wrong slice quietly wrecks retrieval.